In [ ]:
from preprocessing import load_GSE127465, load_nsclc_cosmx
# from transcript_sampling import add_top_st_matches_with_coords
from mapping import integrate_single_st_file_with_harmony
import anndata as ad
import numpy as np
import pandas as pd
from label_transfer import preprocess_and_transfer_cell_types, make_label_correspondence_table, plot_label_correspondence_heatmap
import matplotlib.pyplot as plt
import os

spatial_dir = '/compute/oven-0-13/jwinchel/spatial_adata/'

# sc_data = load_GSE127465('/compute/oven-0-13/jwinchel/scRNA-seq')
sc_data = ad.read_h5ad('/compute/oven-0-13/jwinchel/scRNA-seq/adata/GSE127465_tumor_integrated.h5ad')
# sc_data = sc_data[sc_data.obs['Tissue']=='tumor'] # only tumor data

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:

In [2]:
sc_data.obs.loc['p1t1-bcIIOD-7']

species                                                               human
filename                                   GSM3635278_human_p1t1_raw_counts
sample                                                                 p1t1
GSM_ID                                                           GSM3635278
cell_ID                                                              bcIIOD
Patient                                                                  p1
Tissue                                                                tumor
Barcoding emulsion                                                      p1t
Library                                                                p1t1
Barcode                                                              bcIIOD
Total counts                                                        18765.0
Percent counts from mitochondrial genes                            5.371703
Most likely LM22 cell type                                   Macrophages M0
Major cell t

In [ ]:
# cell_types = sorted(sc_data.obs['cell_type'].unique())
# cell_type_dict = {cell_types[i]: -(i+1) for i in range(len(cell_types))}

# sc_data.uns['cell_type_dict'] = cell_type_dict

# sc_data.obs['split'] = 'train'
# sc_data.obs.loc[sc_data.obs['Patient']=='p2','split'] = 'val'
# sc_data.write_h5ad('/compute/oven-0-13/jwinchel/scRNA-seq/adata/GSE127465_tumor_integrated.h5ad')

In [ ]:

st_data = ad.read_h5ad(os.path.join(spatial_dir,'cosmx_nsclc_Lung9-Rep1_adata_annotated.h5ad'))
st_data.obsm['X_pca_harmony'] = st_data.obsm['X_pca_harmony_Lung_Rep1']

AnnData object with n_obs × n_vars = 91992 × 958
    obs: 'fov', 'cell_ID', 'tissue', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'sdimx', 'sdimy', 'cell_ID_long', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type_orig', 'niche', 'prop_tumor_in_100_neighbors', 'cell_type', 'batch'
    obsm: 'X_pca_harmony_Lung9_Rep1'

In [7]:
st_data = st_data[st_data.obs['cell_type']!='nan']

In [8]:
# Run preprocessing / label transfer
sc_data = preprocess_and_transfer_cell_types(
    sc_data,
    st_data,
    source_major_type_col="Major cell type",
    initial_target_col="cell_type",
    sc_tcell_selector_col="cell_type",
    sc_tcell_selector_value="T cells",
    st_tcell_selector_col="cell_type",
    st_tcell_selector_prefix="T",
    st_merged_label_col="cell_type",
    embedding_key="X_pca_harmony_Lung9_Rep1",
    n_neighbors=30,
    sketch_size=30000,
    random_state=0,
)

sc_data.write_h5ad('/compute/oven-0-13/jwinchel/scRNA-seq/adata/GSE127465_tumor_integrated.h5ad')
# Build and plot correspondence heatmap
ct = make_label_correspondence_table(
    sc_data,
    predicted_col="cell_type",
    original_col="Major cell type",
)

fig, ax = plot_label_correspondence_heatmap(ct)
fig.savefig('/home/jwinchel/Syndri/plots/label_mapping.png')


KeyError: 'X_pca_harmony_Lung9_Rep1'

In [5]:
import os
spatial_files = sorted([f for f in os.listdir(spatial_dir) if f.endswith('annotated.h5ad')])

for f in spatial_files:
    tissue_name = f.split('_')[2].replace('-','_')

    print(tissue_name)

    if f'X_pca_harmony_{tissue_name}' in sc_data.obsm:
        continue
    # batch integration
    sc_data, st_data, cov =  integrate_single_st_file_with_harmony(st_file=os.path.join(spatial_dir, f), sc_data_orig=sc_data,tissue_name=tissue_name)

    # st_data.write_h5ad(os.path.join(spatial_dir, f))
    
    print(sc_data.obsm)



Lung12
Lung13
Lung5_Rep1
Lung5_Rep2
Lung5_Rep3
Lung6
Lung9_Rep1
Lung9_Rep2


In [6]:
st_meta = pd.read_csv('/compute/oven-0-13/jwinchel/NSCLC_ST_cell_metadata.csv')

In [ ]:
# st_meta.loc[(st_meta['Slide_name']=='Lung9')&(st_meta['slide_ID_numeric']==5),'Slide_name'] = 'Lung9_Rep1'
# st_meta.loc[(st_meta['Slide_name']=='Lung9')&(st_meta['slide_ID_numeric']==6),'Slide_name'] = 'Lung9_Rep2'

In [7]:
slide_id = st_meta[['Run_Tissue_name','slide_ID_numeric']].drop_duplicates().values
slide_id = dict(zip(slide_id[:,0], slide_id[:,1]))
slide_id

{'Lung5_Rep1': 1,
 'Lung5_Rep2': 2,
 'Lung5_Rep3': 3,
 'Lung6': 4,
 'Lung9_Rep1': 5,
 'Lung9_Rep2': 6,
 'Lung12': 7,
 'Lung13': 8}

In [8]:
integrated_adatas = []
for pca_key in sc_data.obsm.keys():
    if not pca_key.startswith('X_pca_harmony'):
        continue
    
    tissue_adata = sc_data.copy()
    tissue_name = "_".join(pca_key.split('_')[3:])
    tissue_adata.obsm['X_pca_harmony'] = tissue_adata.obsm[pca_key]
    for key in sc_data.obsm.keys():
        del tissue_adata.obsm[key]
    tissue_adata.obs['harmony_spatial_tissue'] = tissue_name
    tissue_adata.obs['harmony_spatial_id'] = tissue_adata.obs['harmony_spatial_tissue'].map(slide_id)
    print(tissue_adata.obs['harmony_spatial_id'].value_counts())
    integrated_adatas.append(tissue_adata)
    # break
    # integrated_adatas.append(sc_data.obsm[pca_key])

integrated_adata = ad.concat(integrated_adatas, axis=0)



harmony_spatial_id
7    40362
Name: count, dtype: int64
harmony_spatial_id
8    40362
Name: count, dtype: int64
harmony_spatial_id
1    40362
Name: count, dtype: int64
harmony_spatial_id
2    40362
Name: count, dtype: int64
harmony_spatial_id
3    40362
Name: count, dtype: int64
harmony_spatial_id
4    40362
Name: count, dtype: int64
harmony_spatial_id
5    40362
Name: count, dtype: int64
harmony_spatial_id
6    40362
Name: count, dtype: int64


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [9]:
integrated_adata.obs['harmony_spatial_tissue'].value_counts()

harmony_spatial_tissue
Lung12        40362
Lung13        40362
Lung5_Rep1    40362
Lung5_Rep2    40362
Lung5_Rep3    40362
Lung6         40362
Lung9_Rep1    40362
Lung9_Rep2    40362
Name: count, dtype: int64

In [10]:
from mapping import map_sc_subset_to_st

spatial_files = sorted([f for f in os.listdir(spatial_dir) if f.endswith('annotated.h5ad')])

for f in spatial_files:
    tissue_name = f.split('_')[2].replace('-','_')
    print(tissue_name)
    st_data = ad.read_h5ad(os.path.join(spatial_dir, f))
    st_data.obsm['X_pca_harmony'] = st_data.obsm['X_pca_harmony_'+tissue_name]
    del st_data.obsm['X_pca_harmony_'+tissue_name]
    integrated_adata, coverage, match_result = map_sc_subset_to_st(
        sc_data=integrated_adata,
        st_data=st_data,
        tissue_name=tissue_name,
        embedding_key="X_pca_harmony",
        tissue_col="harmony_spatial_tissue",
    )


Lung12


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/Syndri/code/mapping.py:288: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['3554' '40926' '47121' ... '58249' '70581' '66226']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sc_data.obs.iloc[row_idx, sc_data.obs.columns.get_loc(f"{prefix}_{j+1}_id")] = match_result["ids"][:, j]
/home/jwinchel/Syndri/code/mapping.py:288: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['24192' '61240' '37127' ... '67552' '18078' '56630']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sc_data.obs.iloc[row_idx

Lung13


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Lung5_Rep1


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


... (6 outputs left out)

Lung9_Rep1


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Lung9_Rep2


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [11]:
integrated_adata.obs['cell_ID_long'] = integrated_adata.obs['sample'].astype(str) + '-' + integrated_adata.obs['cell_ID'].astype(str) + '-' + integrated_adata.obs['harmony_spatial_id'].astype(str)

In [12]:
integrated_adata.obs.set_index('cell_ID_long', inplace=True)

In [14]:
integrated_adata.obs.columns

Index(['species', 'filename', 'sample', 'GSM_ID', 'cell_ID', 'Patient',
       'Tissue', 'Barcoding emulsion', 'Library', 'Barcode', 'Total counts',
       'Percent counts from mitochondrial genes', 'Most likely LM22 cell type',
       'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells',
       'x_NSCLC_all_cells', 'y_NSCLC_all_cells',
       'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune',
       'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune',
       'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune',
       'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood',
       'Barcode_Library', 'batch', 'cell_type', 'harmony_spatial_tissue',
       'harmony_spatial_id', 'st_match_1_id', 'st_match_1_dist',
       'st_match_1_x', 'st_match_1_y', 'st_match_2_id', 'st_match_2_dist',
       'st_match_2_x', 'st_match_2_y', 'st_match_3_id', 'st_match_3_dist',
       'st_match_3_x', 'st_match_3_y', 'sampled_st_id', 'sampled_st_x',
   

In [18]:
integrated_adata.obs

,species,filename,sample,GSM_ID,cell_ID,Patient,Tissue,Barcoding emulsion,Library,Barcode,...,st_match_2_x,st_match_2_y,st_match_3_id,st_match_3_dist,st_match_3_x,st_match_3_y,sampled_st_id,sampled_st_x,sampled_st_y,selected_match_idx
cell_ID_long,,,,,,,,,,,,,,,,,,,,,
p1t1-bcIIOD-7,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcIIOD,p1,tumor,p1t,p1t1,bcIIOD,...,-1574.777778,-152274.333333,47121,6.276097,10118.666667,-142722.333333,3554,2592.444444,-157186.333333,0.0
p1t1-bcHTNA-7,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcHTNA,p1,tumor,p1t,p1t1,bcHTNA,...,13693.888889,-141566.333333,31051,4.915611,12566.888889,-150002.333333,31051,12566.888889,-150002.333333,2.0
p1t1-bcDLAV-7,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcDLAV,p1,tumor,p1t,p1t1,bcDLAV,...,1645.444444,-148590.333333,9605,6.599017,15332.888889,-158683.333333,37127,1645.444444,-148590.333333,1.0
p1t1-bcHNVA-7,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcHNVA,p1,tumor,p1t,p1t1,bcHNVA,...,7305.666667,-140200.333333,31051,4.917272,12566.888889,-150002.333333,31051,12566.888889,-150002.333333,2.0
p1t1-bcALZN-7,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcALZN,p1,tumor,p1t,p1t1,bcALZN,...,12362.888889,-144937.333333,66539,6.210737,1904.444444,-137994.333333,14538,5846.444444,-154282.333333,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
p7t2-bcFAUI-6,human,GSM3635302_human_p7t2_raw_counts,p7t2,GSM3635302,bcFAUI,p7,tumor,p7t,p7t2,bcFAUI,...,-17567.000000,-16026.777778,775797,5.445290,-1250.333333,-2049.777778,775797,-1250.333333,-2049.777778,2.0
p7t2-bcFHTY-6,human,GSM3635302_human_p7t2_raw_counts,p7t2,GSM3635302,bcFHTY,p7,tumor,p7t,p7t2,bcFHTY,...,-13595.000000,3049.222222,724722,6.701613,-12189.777778,-10789.777778,724722,-12189.777778,-10789.777778,2.0
p7t2-bcBHQQ-6,human,GSM3635302_human_p7t2_raw_counts,p7t2,GSM3635302,bcBHQQ,p7,tumor,p7t,p7t2,bcBHQQ,...,-16425.000000,-1657.777778,734831,5.143020,-15624.000000,-7868.777778,764796,-16425.000000,-1657.777778,1.0


In [17]:
import sys
integrated_adata.write_h5ad(os.path.join('/compute/oven-0-13/jwinchel/scRNA-seq/adata/','GSE127465_tumor_integrated.h5ad'))